## LORA training/testing pipeline — Task 1 (Risk Clause Recognition), Hard Negatives

This notebook implements the fine-tuning pipeline for **Task 1: binary clause identification**, following [TASK1_HARD_NEGATIVES_PLAN.md](docs/TASK1_HARD_NEGATIVES_PLAN.md) (Option A — Hard Negatives).

Task 1 = given a single contract clause excerpt, decide whether it is an instance of a specific risk clause category (`Yes`) or not (`No`) across the 32 Yes/No categories.

**The fix this notebook applies (closing the leak):** in the raw CSV, a category's input text is non-empty *exactly when* the answer is `Yes`. Training naively on that lets the model cheat — "any real text → Yes, placeholder → No" — without ever reading a clause. Instead, every `No` example **borrows a real clause from a different category in the same contract** (a *hard negative*), so both `Yes` and `No` inputs are genuine legal text and the model must actually recognize the clause type.

**Note:** this notebook loads the *sampled* CSV (`master_clauses_cleaned_sampled.csv`, ~100 records) for quick iteration instead of the full dataset.

# Step 1 : Load data (sampled master_clauses file from CUAD)

Dataset Description Summarized : 

1. Columns NOT ending in "Answer" (Context Columns)
- Role: These columns contain the text context (the actual excerpt or "clause") extracted from the contract.
- Content: A string of text directly from the contract that is responsive to a specific category.
- Purpose: This serves as the "evidence" or the "source passage" that justifies a specific determination.
- Handling of Omissions: If parts of the text are irrelevant, they may be replaced with <omitted>.

2. Columns ending in "Answer" (Label Columns)
- Role: These columns contain the derived human-input answers based on the text context found in the corresponding Context column.
- Content:
- For "Yes/No" Categories (32 types): The value is "Yes" if the clause exists, or "No" if no string was found. (e.g., Termination for Convenience).
- For Extraction Categories (Task 2): The value is a normalized string representing a specific entity, date, or number.
- Purpose: This is the "ground truth" or "label" for the machine learning task.

In [ ]:
import pandas as pd
import json
from pathlib import Path
import csv
import re
from sklearn.model_selection import train_test_split

In [ ]:
CUAD_PATH = Path('data/CUAD_v1')
# Step 1: load the cleaned CSV.
# The plan calls for the full master_clauses_cleaned.csv (510 contracts); for quick
# iteration we use the sampled file (~100 records) — the only deviation from the plan.
# MASTER_CLAUSES_PATH = CUAD_PATH/'master_clauses_cleaned.csv'        # full dataset
MASTER_CLAUSES_PATH = CUAD_PATH/'master_clauses_cleaned_sampled.csv'  # ~100 records, quick iteration

try:
    # Read the file manually using the CSV module first to handle inconsistencies
    data = []
    with open(MASTER_CLAUSES_PATH, 'r', encoding='utf-8', errors='replace') as f:
        # Use csv.Sniffer to deduce format if possible, or enforce standard strictness
        reader = csv.DictReader(f) 
        for i, row in enumerate(reader):
            data.append(row)

    # Convert the list of dicts to a DataFrame
    df = pd.DataFrame(data)

    print(f"Data Loaded Successfully via CSV module.")
    print(f"Total Contracts: {len(df)}")
    print(df.head(3))

except Exception as e:
    print(f"Error: {e}")

In [ ]:
df.head(3)

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove special characters but keep spaces
    return re.sub(r'[^a-zA-Z0-9\s]', '', text)

# Clean column names
df.columns = [clean_text(col).strip() for col in df.columns]
for col in df.columns:
    print(col)

In [ ]:
new_columns = {}
for col in df.columns:
    print(f"Processing column: '{col}'")
    if "Answer" in col:
            # Remove "Answer" from the string and append "_Answer" at the end
            new_columns[col] = f"{col.replace('Answer', '').strip()}_Answer"

df = df.rename(columns=new_columns)

In [ ]:
for col in df.columns:
    print(col)

# Step 2 (cont.) : Restrict to the 32 Task 1 categories

Per the plan, the load / column-clean / `_Answer`-rename cells above are left unchanged. Here we derive the Task 1 category list by excluding the Task 2 entity-extraction fields, so Task 2 fields never enter Task 1 training.

In [ ]:
task2_categories = [
    "Filename", "Document Name", "Parties", "Agreement Date", "Effective Date",
    "Expiration Date", "Renewal Term", "Notice Period To Terminate Renewal",
    "Governing Law", "Warranty Duration",
]
task1_categories = [
    col for col in df.columns
    if not col.endswith("_Answer") and col.strip() != ""
    and col not in task2_categories
    and f"{col}_Answer" in df.columns
]
assert len(task1_categories) == 32, f"Expected 32 Task 1 categories, got {len(task1_categories)}"
print(f"{len(task1_categories)} Task 1 categories:")
for c in task1_categories:
    print(" -", c)

In [ ]:
def save_jsonl(data, filename):
    with open(filename, 'w') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')

# Step 3 & 4 : Build binary (Yes/No) examples with **hard negatives**, split by contract ⭐

This is the real fix. For each contract row we gather **all clauses actually present** in that contract, keyed by category. Then for each of the 32 categories we build one example:

- **Yes** → use that category's own real clause text.
- **No** → randomly borrow a real clause from a **different** present category (a *hard negative*). If the contract has no other clause to borrow, skip the example.

So both `Yes` and `No` inputs are genuine legal text — the model can no longer cheat off a placeholder, and the only way to answer is to recognize the clause type. The instruction wording is *"Is the following contract text a `...` clause?"* because we now feed a single clause excerpt.

The split is done on **contracts (df rows) first** (Step 4), then examples are built from each side, to prevent a contract leaking across train/val.

In [ ]:
import random
random.seed(42)

def to_binary(answer):
    return "No" if (pd.isna(answer) or str(answer).strip().lower() == "no") else "Yes"

def nonempty_context(row, cat):
    v = row.get(cat)
    return str(v).strip() if pd.notna(v) and str(v).strip() else None

def build_examples(frame):
    rows = []
    for _, row in frame.iterrows():
        # every clause actually present in THIS contract, keyed by category
        present = {c: nonempty_context(row, c)
                   for c in task1_categories if nonempty_context(row, c)}
        for category in task1_categories:
            if to_binary(row[f"{category}_Answer"]) == "Yes":
                text, label = present[category], "Yes"
            else:
                # hard negative: a real clause from a DIFFERENT category
                others = [t for c, t in present.items() if c != category]
                if not others:
                    continue                      # nothing to borrow -> skip
                text, label = random.choice(others), "No"
            rows.append({
                "instruction": f'Is the following contract text a "{category}" clause? Answer strictly "Yes" or "No".',
                "category": category,
                "input": text,
                "output": label,
            })
    return rows

# Step 4: split by CONTRACT first, then build examples from each side.
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_data = build_examples(train_df)
val_data = build_examples(val_df)

print(f"Contracts — train: {len(train_df)}, val: {len(val_df)}")
print(f"Examples  — train: {len(train_data)}, val: {len(val_data)}")

# Step 5 : Balance classes on the **train** split only

Hard negatives can still be a minority/majority depending on how many clauses each contract has, and without balancing the model drifts to always answering the majority class. Downsample the majority class toward ~1:1 on **train only**; leave `val_data` at its natural distribution so validation metrics stay honest.

In [ ]:
from collections import defaultdict

def balance(data, ratio=1.0, seed=42):
    rng = random.Random(seed)
    pos = [e for e in data if e["output"] == "Yes"]
    neg = [e for e in data if e["output"] == "No"]
    keep_neg = min(len(neg), int(len(pos) * ratio))
    neg = rng.sample(neg, keep_neg)
    out = pos + neg
    rng.shuffle(out)
    return out

train_data = balance(train_data, ratio=1.0)   # train only — val untouched
print(f"After balancing — train: {len(train_data)} examples")

# Step 6a : Sanity checks — class counts + verify hard negatives are real text

Two quick checks before saving:
1. Print the final `Yes`/`No` counts (train is balanced; val is natural).
2. **Inspect a few `No` examples** — their `input` must be *real clause text borrowed from another category*, never the old `[No matching clause excerpt found...]` placeholder. If a `No` input is a placeholder, the leak isn't closed.

The headline per-class precision / recall / F1 + confusion matrix is computed **after training** in Step 6b.

In [ ]:
from collections import Counter

def label_counts(data):
    return Counter(ex["output"] for ex in data)

print("Train label counts:", dict(label_counts(train_data)))
print("Val   label counts:", dict(label_counts(val_data)))

# Verify hard negatives: every `No` input must be REAL clause text, not a placeholder.
print("\nSample of `No` examples (inputs must be real borrowed clause text):")
no_examples = [ex for ex in train_data if ex["output"] == "No"]
for ex in no_examples[:3]:
    print(f"\n  category : {ex['category']}")
    print(f"  input    : {ex['input'][:200]}{'...' if len(ex['input']) > 200 else ''}")

assert all("[No matching clause excerpt found" not in ex["input"] for ex in no_examples), \
    "Found placeholder text in a No example — the leak is NOT closed."
print("\nOK — no placeholder strings found in `No` inputs.")

# Save examples to JSONL (ensure dirs exist; save paths == load paths)

In [ ]:
CUAD_TRAIN_PATH = CUAD_PATH/'train'
CUAD_VALIDATION_PATH = CUAD_PATH/'validation'

# Ensure the train/ and validation/ directories exist before saving.
CUAD_TRAIN_PATH.mkdir(parents=True, exist_ok=True)
CUAD_VALIDATION_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
save_jsonl(train_data, CUAD_TRAIN_PATH/'cuad_train.jsonl')
save_jsonl(val_data, CUAD_VALIDATION_PATH/'cuad_validation.jsonl')
print(f"Saved {len(train_data)} training samples and {len(val_data)} validation samples.")

# Step 7 & 8 : QLoRA fine-tuning with completion-only loss

Two changes versus the original pipeline:

- **Step 7 — completion-only loss:** the answer is a single token (`Yes`/`No`), so computing loss over the whole prompt lets the gradient be dominated by reproducing the clause text and drowns out the decision signal. `DataCollatorForCompletionOnlyLM` masks everything before `### Response:\n`, so only the answer contributes to the loss.
- **Step 8 — config tidy-ups:** prefer `bf16` over `fp16` for stability, widen LoRA targets to `["q_proj","k_proj","v_proj","o_proj"]`, and lower `max_seq_length` to `1024` (single clauses are short → faster, less memory).

The prompt template is unchanged (`### Instruction / ### Input / ### Response`); `load_dataset` points at the same JSONL files written in the save cell above.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# 1. Configuration
model_name = "meta-llama/Meta-Llama-3-8B" # or "mistralai/Mistral-7B-v0.1"
new_model_name = "llama-3-cuad-finetune"

# 2. QLoRA Config (4-bit loading to fit on consumer GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # Step 8: bf16 for stability
)

# 3. Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False # Silence warnings during training

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16

# 5. Load Dataset (load the same files that were saved)
dataset = load_dataset("json", data_files={
    "train":      str(CUAD_TRAIN_PATH / "cuad_train.jsonl"),
    "validation": str(CUAD_VALIDATION_PATH / "cuad_validation.jsonl"),
})

# 6. LoRA Configuration
peft_config = LoraConfig(
    r=16,       # Rank (Higher = more parameters to train, 16-64 is standard)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # Step 8: wider targets for a slightly stronger adapter
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# 7. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,           # 1 epoch is often enough for SFT on small datasets
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    weight_decay=0.001,
    bf16=True,                    # Step 8: bf16 over fp16 for stability
    logging_steps=25,
    save_steps=100,
    optim="paged_adamw_32bit",    # Syllabus optimization
)

# Step 7: completion-only loss — mask everything before "### Response:\n" so only
# the Yes/No answer contributes to the loss, not the clause text being echoed back.
collator = DataCollatorForCompletionOnlyLM(
    response_template="### Response:\n",
    tokenizer=tokenizer,
)

# 8. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    data_collator=collator,  # Step 7: completion-only loss
    max_seq_length=1024,     # Step 8: single clauses are short
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=lambda example: [
        f"### Instruction:\n{inst}\n\n### Input:\n{inp}\n\n### Response:\n{out}"
        for inst, inp, out in zip(example['instruction'], example['input'], example['output'])
    ]
)

# 9. Train and Save
print("Starting training...")
trainer.train()
trainer.model.save_pretrained(new_model_name)
print(f"Model saved to {new_model_name}")

# Step 6b : Evaluate on validation — per-class precision / recall / F1

Under class imbalance, accuracy and loss are meaningless ("always No" can score 80%+). Generate a `Yes`/`No` prediction for every validation example and report **per-class precision / recall / F1 + a confusion matrix** with `sklearn.metrics.classification_report`.

**How to know the fix worked:** a model that ignores the input should now score ~50%, not ~100%. If validation F1 is near-perfect immediately, re-inspect the inputs — the leak may not be fully closed.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Use cache for faster generation at inference time.
model.config.use_cache = True
model.eval()

def predict(example):
    prompt = (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Input:\n{example['input']}\n\n"
        f"### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return "Yes" if "yes" in decoded.strip().lower() else "No"

y_true = [ex["output"] for ex in val_data]
y_pred = [predict(ex) for ex in val_data]

print("Per-class precision / recall / F1 on validation:\n")
print(classification_report(y_true, y_pred, labels=["Yes", "No"], zero_division=0))

print("Confusion matrix (rows = true [Yes, No], cols = pred [Yes, No]):")
print(confusion_matrix(y_true, y_pred, labels=["Yes", "No"]))